# Negacyclic NTT in JAX — Beginner-Friendly Tutorial

> **ECE-9413 Assignment 1 companion notebook**  
> Covers every concept from scratch: no prior knowledge of FFT, number theory, or JAX required.

---

## What you will learn

| Section | What you build |
|---|---|
| 1. From DFT to NTT | Why NTT is FFT over a finite field |
| 2. Modular arithmetic | `mod_add`, `mod_sub`, `mod_mul` in JAX |
| 3. Roots of unity mod q | How `psi` is found and what it guarantees |
| 4. Negacyclic twist | What `x^N + 1` means and the psi-twist trick |
| 5. Butterfly & Cooley-Tukey | The recursive divide-and-conquer structure |
| 6. Stockham NTT | The GPU-friendly out-of-place variant |
| 7. Twiddle & psi_powers tables | What `provided.precompute_tables` builds |
| 8. Full NTT implementation | Complete working JAX code |
| 9. GPU/TPU optimisation | Montgomery form, memory layout, JIT, vmap |
| 10. prepare_tables + benchmarking | Precompute once, time accurately |

---
## Section 0 — Setup

In [ ]:
# Install dependencies — change 'cuda12' to 'cpu' if no GPU
!pip install -q --upgrade "jax[cuda12]" jaxlib sympy

import jax
import jax.numpy as jnp
import numpy as np

jax.config.update("jax_enable_x64", True)  # needed for 64-bit intermediates

print("JAX version :", jax.__version__)
print("Devices     :", jax.devices())
print("Backend     :", jax.default_backend())

---
## Section 1 — From DFT to NTT: the Core Analogy

### The Discrete Fourier Transform (DFT)

The DFT of a sequence `x[0..N-1]` is:

```
y[k] = Σ_{n=0}^{N-1}  x[n] · ω^{k·n}     ω = e^{2πi/N}
```

Here `ω` is a primitive N-th root of unity in the **complex numbers** (ω^N = 1, ω^j ≠ 1 for 0 < j < N).

### The Number Theoretic Transform (NTT)

The NTT replaces `C` with a **finite field** `Z/qZ` (integers mod a prime `q`).  
Everything is exactly the same formula, but:
- All arithmetic is `mod q` (no floating-point errors!).
- `ω` is a primitive N-th root of unity **modulo q** (ω^N ≡ 1 mod q, ω^j ≢ 1 for 0 < j < N).

```
y[k] = Σ_{n=0}^{N-1}  x[n] · ω^{k·n}  (mod q)
```

### Why NTT over DFT for cryptography?

| | DFT | NTT |
|---|---|---|
| Arithmetic | Floating-point (inexact) | Modular integers (exact) |
| Output type | Complex numbers | Integers in [0, q) |
| Use case | Signal processing, audio | Polynomial multiplication mod q |
| Lattice crypto | Not usable | Core primitive |

NTT is the backbone of modern lattice-based cryptography (NTRU, Kyber, Dilithium).

In [ ]:
import numpy as np

# ── DFT on 4 elements ───────────────────────────────────────────────────────
x = [1, 2, 3, 4]
N = len(x)

# DFT using complex roots of unity
omega_c = np.exp(2j * np.pi / N)
y_dft = [sum(x[n] * omega_c**(k*n) for n in range(N)) for k in range(N)]
print("DFT output (complex):")
for k, v in enumerate(y_dft):
    print(f"  y[{k}] = {v.real:.1f} + {v.imag:.1f}j")

# ── NTT on 4 elements mod 17 ────────────────────────────────────────────────
# Need q where (q-1) divisible by N=4. q=17: q-1=16, 16%4=0. Yes.
# Primitive 4th root of unity mod 17:
#   omega^4 = 1 mod 17, omega^2 ≠ 1, omega^1 ≠ 1
#   Try omega=4: 4^1=4, 4^2=16, 4^4=256%17=256-15*17=256-255=1. Yes!
q = 17
omega = 4  # primitive 4th root of unity mod 17

print(f"\nVerify omega={omega} is a primitive 4th root of unity mod q={q}:")
for p in [1, 2, 4]:
    print(f"  omega^{p} mod {q} = {pow(omega, p, q)}")   # should be 4, 16, 1

y_ntt = [sum(x[n] * pow(omega, k*n, q) for n in range(N)) % q for k in range(N)]
print(f"\nNTT output (integers mod {q}): {y_ntt}")
print("Notice: all outputs are exact integers in [0, q), no floating-point!")

---
## Section 2 — Modular Arithmetic Primitives

Every NTT operation reduces to three primitives: add, subtract, multiply — all mod q.

### The overflow problem

The assignment uses **31-bit primes** (up to ~2.1 billion) stored in `uint32`.  
- `a + b` can be up to ~4.2e9, which **overflows** uint32 (max ~4.29e9) — marginal, must promote.
- `a * b` can be up to ~4.4e18, which **definitely overflows** uint32. Must promote to uint64.

Strategy: **promote to uint64 for the operation, reduce mod q, demote back to uint32**.

In [ ]:
import jax.numpy as jnp

# ── mod_add: (a + b) mod q ───────────────────────────────────────────────────
def mod_add(a, b, q):
    """
    (a + b) mod q, elementwise.
    Promote to uint64 to avoid overflow on max 31-bit prime.
    """
    return ((a.astype(jnp.uint64) + b.astype(jnp.uint64)) % jnp.uint64(q)).astype(jnp.uint32)


# ── mod_sub: (a - b) mod q ───────────────────────────────────────────────────
def mod_sub(a, b, q):
    """
    (a - b) mod q, elementwise.
    Add q before subtracting to ensure the result stays non-negative
    (unsigned integers cannot represent negative values).
    """
    q64 = jnp.uint64(q)
    return ((a.astype(jnp.uint64) + q64 - b.astype(jnp.uint64)) % q64).astype(jnp.uint32)


# ── mod_mul: (a * b) mod q ───────────────────────────────────────────────────
def mod_mul(a, b, q):
    """
    (a * b) mod q, elementwise.
    For 31-bit q: max(a*b) = (2^31)^2 = 2^62, which fits in uint64 (max 2^64-1).
    """
    return ((a.astype(jnp.uint64) * b.astype(jnp.uint64)) % jnp.uint64(q)).astype(jnp.uint32)


# ── Verify on tiny example ───────────────────────────────────────────────────
Q = jnp.uint32(17)
a = jnp.array([5, 10, 16], dtype=jnp.uint32)
b = jnp.array([3,  9,  3], dtype=jnp.uint32)

print(f"a         = {a.tolist()}")
print(f"b         = {b.tolist()}")
print(f"mod_add   = {mod_add(a, b, Q).tolist()}   # [8, 2, 2]")
print(f"mod_sub   = {mod_sub(a, b, Q).tolist()}   # [2, 1, 13]")
print(f"mod_mul   = {mod_mul(a, b, Q).tolist()}   # [15, 5, 14]")

---
## Section 3 — Roots of Unity Modulo q

Just like `e^{2πi/N}` is the primitive N-th root of unity in `C`, we need a number `ω mod q` such that:
- `ω^N ≡ 1 (mod q)` — it cycles back to 1 after N steps.
- `ω^j ≢ 1 (mod q)` for any `0 < j < N` — it doesn't cycle early.

### When does such an ω exist?

A primitive N-th root of unity mod q exists **if and only if N divides (q - 1)**. This is why the assignment's `generate_ntt_modulus` searches for primes where `(q - 1) % N == 0`.

### Finding ω: generator method

1. Find a **generator** `g` of `(Z/qZ)*` — a number whose powers cycle through all nonzero residues mod q.
2. Set `ω = g^{(q-1)/N} mod q`. Now `ω^N = g^{q-1} = 1 (mod q)` (Fermat's little theorem).

In [ ]:
from sympy import isprime

def find_generator(q):
    """Find a primitive root (generator) of (Z/qZ)*."""
    phi = q - 1  # for prime q, |group| = q-1
    # Factorize phi
    factors = set()
    n = phi
    d = 2
    while d * d <= n:
        if n % d == 0:
            factors.add(d)
            while n % d == 0:
                n //= d
        d += 1
    if n > 1:
        factors.add(n)
    # g is a generator if g^(phi/p) ≠ 1 mod q for all prime factors p of phi
    for g in range(2, q):
        if all(pow(g, phi // p, q) != 1 for p in factors):
            return g
    raise ValueError("no generator found")


def find_primitive_root_of_unity(order, q):
    """Find a primitive `order`-th root of unity mod q."""
    assert (q - 1) % order == 0, f"{order} must divide q-1={q-1}"
    g = find_generator(q)
    return pow(g, (q - 1) // order, q)


# ── Example: N=8, find a suitable prime q and primitive 8th root of unity ────
N = 8

# Smallest prime q where (q-1) % 8 == 0:  q=17 (16%8=0, yes)
q = 17
print(f"q={q}, q-1={q-1}, (q-1) % N={q-1} % {N} = {(q-1) % N}")
assert (q - 1) % N == 0

omega = find_primitive_root_of_unity(N, q)
print(f"\nPrimitive {N}-th root of unity mod {q}: ω = {omega}")

print(f"\nPowers of ω={omega} mod {q}:")
for k in range(N + 1):
    print(f"  ω^{k} = {pow(omega, k, q)}", end="  ")
    if (k + 1) % 4 == 0:
        print()

print(f"\nVerification:")
print(f"  ω^N = ω^{N} = {pow(omega, N, q)}  (must be 1)")
print(f"  ω^(N/2) = ω^{N//2} = {pow(omega, N//2, q)}  (must not be 1, it's -1 ≡ {q-1} mod {q})")

---
## Section 4 — The Negacyclic Twist: What is `x^N + 1`?

### Two flavours of NTT

| Variant | Polynomial ring | Root used | Property |
|---|---|---|---|
| **Cyclic NTT** | `Z[x] / (x^N - 1)` | ω : `ω^N = 1` | Standard DFT analogue |
| **Negacyclic NTT** | `Z[x] / (x^N + 1)` | ψ : `ψ^N = -1` | Used in lattice crypto |

The negacyclic ring `Z[x]/(x^N + 1)` is preferred in cryptography because:
- N can be a power of two (good for FFT).
- `x^N + 1` is irreducible over the integers → no zero divisors → better security.

### The psi (ψ) root

For negacyclic NTT of size N we need `ψ`, a **primitive 2N-th root of unity** mod q:
- `ψ^{2N} ≡ 1 (mod q)`
- `ψ^N ≡ -1 (mod q)` ← this is the key "negacyclic" property
- `ω = ψ^2` is then a primitive N-th root of unity

### The twist trick

A negacyclic NTT of size N is equivalent to:
1. **Twist**: multiply each input coefficient `x[n]` by `ψ^n`.
2. **Cyclic NTT** with root `ω = ψ²`.

```
negacyclic_NTT(x)[k] = cyclic_NTT(x' where x'[n] = x[n] * ψ^n)[k]
```

This means the negacyclic NTT formula from the assignment:
```
y[k] = Σ x[n] · ψ^{(2k+1)·n}  =  Σ (x[n]·ψ^n) · (ψ²)^{k·n}
              ↑ negacyclic            ↑ twist      ↑ cyclic with ω=ψ²
```

In [ ]:
from sympy.discrete.transforms import ntt as sympy_ntt

# ── Build parameters ─────────────────────────────────────────────────────────
N = 4
# Need q where (q-1) divisible by 2N=8.  q=17: 16%8=0, yes.
q = 17

# psi: primitive 2N=8-th root of unity mod q
psi = find_primitive_root_of_unity(2 * N, q)
omega = pow(psi, 2, q)  # omega = psi^2 is a primitive N-th root

print(f"N={N}, q={q}")
print(f"ψ = {psi}")
print(f"  ψ^N  = ψ^{N}  = {pow(psi, N, q)}  (should be {q-1} = -1 mod {q})")
print(f"  ψ^2N = ψ^{2*N} = {pow(psi, 2*N, q)}  (should be 1)")
print(f"ω = ψ² = {omega}")
print(f"  ω^N  = ω^{N}  = {pow(omega, N, q)}  (should be 1)")

# ── Test input ───────────────────────────────────────────────────────────────
x = [1, 2, 3, 4]

# ── Method A: direct negacyclic formula ─────────────────────────────────────
y_direct = [
    sum(x[n] * pow(psi, (2*k+1)*n, q) for n in range(N)) % q
    for k in range(N)
]

# ── Method B: twist + cyclic NTT ─────────────────────────────────────────────
psi_powers_list = [pow(psi, n, q) for n in range(N)]
x_twisted = [(x[n] * psi_powers_list[n]) % q for n in range(N)]
y_twist = sympy_ntt(x_twisted, q)  # cyclic NTT with omega = psi^2
y_twist = [v % q for v in y_twist]

print(f"\nInput x = {x}")
print(f"Direct negacyclic formula : {y_direct}")
print(f"Twist + cyclic NTT        : {y_twist}")
print(f"Methods match: {y_direct == y_twist}")

---
## Section 5 — The Butterfly: How NTT Achieves O(N log N)

### The naive DFT is O(N²)

Computing `y[k] = Σ x[n] · ω^{kn}` for all k directly takes O(N) multiplications per output, so O(N²) total.

### The Cooley-Tukey divide-and-conquer

Split the input into **even** and **odd** indices:
```
y[k] = Σ_{n even} x[n]·ω^{kn}  +  Σ_{n odd} x[n]·ω^{kn}
     = NTT(x_even)[k]  +  ω^k · NTT(x_odd)[k]
```

This halves the problem recursively, giving **O(N log N)**. The fundamental operation is the **butterfly**:

```
given (u, v) and twiddle factor W:
  output_top    = u + W·v   (mod q)
  output_bottom = u - W·v   (mod q)
```

### Visualising one butterfly

In [ ]:
import jax.numpy as jnp

def butterfly(u, v, W, q):
    """
    The fundamental NTT butterfly.

    u, v: input values (scalars or arrays)
    W   : twiddle factor = omega^k mod q
    q   : prime modulus

    Returns:
        (top, bot) = (u + W*v, u - W*v)  mod q
    """
    Wv  = mod_mul(v, W, q)      # W * v  mod q
    top = mod_add(u, Wv, q)     # u + Wv mod q
    bot = mod_sub(u, Wv, q)     # u - Wv mod q
    return top, bot


# ── Verify: one butterfly step for N=2 NTT ────────────────────────────────────
# NTT of [x0, x1] with omega (primitive 2nd root of unity):
#   y[0] = x0 + omega^0 * x1 = x0 + x1
#   y[1] = x0 + omega^1 * x1 = x0 - x1  (since omega^1 = -1 for N=2)
#
# For q=17: primitive 2nd root of unity is -1 = 16
Q = jnp.uint32(17)
x0 = jnp.uint32(3)
x1 = jnp.uint32(5)
W0 = jnp.uint32(1)   # omega^0 = 1  (twiddle for y[0])
W1 = jnp.uint32(16)  # omega^1 = -1 = 16 mod 17  (twiddle for y[1])

y0, _ = butterfly(x0, x1, W0, Q)
y1, _ = butterfly(x0, x1, W1, Q)
print(f"NTT([{int(x0)}, {int(x1)}]) mod {int(Q)} = [{int(y0)}, {int(y1)}]")
print(f"Expected: [{(3+5)%17}, {(3-5)%17}] = [8, 15]")

# ── Butterfly on arrays (this is where GPU wins big) ─────────────────────────
# Instead of one pair (u, v), we process N/2 pairs simultaneously.
# Each pair is independent — perfect for parallel hardware.
u = jnp.array([3, 7, 1, 9], dtype=jnp.uint32)   # N/2 = 4 "upper" values
v = jnp.array([5, 2, 8, 3], dtype=jnp.uint32)   # N/2 = 4 "lower" values
W = jnp.array([1, 4, 16, 13], dtype=jnp.uint32)  # 4 twiddle factors

top, bot = butterfly(u, v, W, Q)
print(f"\nVectorised butterfly (all 4 pairs at once):")
print(f"  top = {top.tolist()}")
print(f"  bot = {bot.tolist()}")
print("GPU processes all N/2 pairs in parallel — this is the key parallelism.")

---
## Section 6 — Stockham NTT: The GPU-Friendly Algorithm

### Why not plain Cooley-Tukey?

The classical Cooley-Tukey algorithm is **in-place**: it reads and writes the same array. This creates **bank conflicts** and **non-coalesced memory accesses** on GPUs — hardware parallelism is wasted waiting for memory.

### Stockham: out-of-place, always coalesced

Stockham NTT uses **two buffers** (ping-pong). Each stage reads from buffer A and writes to buffer B contiguously. The access pattern is:
- Always reads/writes consecutive memory addresses.
- Perfectly coalesced on GPU (all threads in a warp hit the same cache line).

### Stage structure

For a size-N NTT with `log2(N)` stages, at stage `s` (0-indexed):
- `span = 2^s`   (current "half-block" size)
- `num_blocks = N / (2 * span)` blocks
- Each block has `2 * span` elements
- Within each block, element `j` (0 ≤ j < span) uses twiddle `twiddles[span + j]`

The `twiddles` array from `provided.precompute_tables` is indexed exactly this way.

In [ ]:
import numpy as np

# ── Visualise the Stockham twiddle layout for N=8 ────────────────────────────
N = 8
q_val = 17   # toy prime (not the NTT prime for N=8, just for illustration)

# Let's use a real NTT prime for N=8
# We need q where (q-1) % 16 == 0 (because negacyclic uses 2N=16)
# q=97: 96 % 16 = 0. Yes.
q_val = 97
psi_val = find_primitive_root_of_unity(2 * N, q_val)
omega_val = pow(psi_val, 2, q_val)

print(f"N={N}, q={q_val}, ψ={psi_val}, ω=ψ²={omega_val}")
print(f"ψ^N = {pow(psi_val, N, q_val)} (should be {q_val-1} = -1 mod {q_val})")

stages = N.bit_length() - 1  # = log2(N) = 3
print(f"\nNumber of stages: {stages} (= log2({N}))")
print(f"\nStockham twiddle table layout:")
print(f"  Index range [span, 2*span) holds twiddles for stage s where span=2^s")
print()

twiddles = [1] * N
for s in range(stages):
    span   = 1 << s
    stride = N // (2 * span)   # = N / (2 * 2^s)
    step   = pow(omega_val, stride, q_val)  # omega^stride
    cur    = 1
    print(f"  Stage s={s}: span={span}, stride={stride}, step=ω^{stride}={step}")
    for j in range(span):
        twiddles[span + j] = cur
        print(f"    twiddles[{span+j}] = ω^({stride}*{j}) = {cur}")
        cur = (cur * step) % q_val

print(f"\nFull twiddle array: {twiddles}")

---
## Section 7 — The psi_powers and twiddles Tables

The provided `precompute_tables(N, q, psi)` function builds two arrays:

### `psi_powers[n]` = ψ^n mod q
- Used for the **negacyclic twist**: multiply `x[n]` by `psi_powers[n]` before the cyclic NTT.
- Length N.

### `twiddles[span + j]` = ω^{j × stride} mod q
- Used for the **Stockham butterfly** at each stage.
- `ω = ψ²`, `stride = N / (2 × span)`, `span = 2^s` for stage `s`.
- Length N (indices 0..N-1, where index 0 is unused).

Let's verify these match the `provided.py` implementation:

In [ ]:
import numpy as np

def my_precompute_tables(N, q, psi):
    """
    Reimplement provided.precompute_tables from scratch for understanding.

    Returns:
        psi_powers: array of length N, psi_powers[n] = psi^n mod q
        twiddles  : array of length N, twiddles[span+j] = omega^(j*stride) mod q
                    where omega = psi^2, span = 2^s, stride = N/(2*span)
    """
    q, psi = int(q), int(psi)
    omega = pow(psi, 2, q)

    # --- psi_powers[n] = psi^n ---
    psi_powers = []
    cur = 1
    for n in range(N):
        psi_powers.append(cur)
        cur = (cur * psi) % q

    # --- twiddles[span + j] = omega^(j * stride) ---
    twiddles = [1] * N
    stages = N.bit_length() - 1  # log2(N)
    for s in range(stages):
        span   = 1 << s
        stride = N // (2 * span)
        step   = pow(omega, stride, q)  # omega^stride
        cur    = 1
        for j in range(span):
            twiddles[span + j] = cur
            cur = (cur * step) % q

    return np.array(psi_powers, dtype=np.uint32), np.array(twiddles, dtype=np.uint32)


# ── Test against the provided implementation ─────────────────────────────────
N_test = 16
q_test = 97   # a small prime where (q-1) % 2N = 0: 96 % 32 = 0
psi_test = find_primitive_root_of_unity(2 * N_test, q_test)

my_pp, my_tw = my_precompute_tables(N_test, q_test, psi_test)

print(f"N={N_test}, q={q_test}, ψ={psi_test}")
print(f"\npsi_powers: {my_pp.tolist()}")
print(f"  Verify: psi_powers[0]=1, psi_powers[1]={psi_test}, psi_powers[N//2]={pow(psi_test, N_test//2, q_test)} (should be ψ^{N_test//2})")
print(f"  ψ^N = ψ^{N_test} = {pow(psi_test, N_test, q_test)} (should be {q_test-1} = -1 mod {q_test})")
print(f"\ntwiddles: {my_tw.tolist()}")
print(f"  twiddles[1] = {my_tw[1]} (stage 0, span=1, only twiddle = ω^(N/2))")
print(f"  ω^(N/2) = {pow(pow(psi_test,2,q_test), N_test//2, q_test)} (should match)")

---
## Section 8 — Full NTT Implementation

Now we put it all together. The algorithm is:

```
1. TWIST: multiply x[n] by psi_powers[n]  → x_twisted
2. CYCLIC NTT (Stockham) on x_twisted:
   for each stage s = 0, 1, ..., log2(N)-1:
       span   = 2^s
       for each block:
           for j in 0..span-1:
               u = current[block_start + j]
               v = current[block_start + span + j]
               W = twiddles[span + j]
               next[...] = butterfly(u, v, W)
3. Return result
```

### Key JAX insight: avoid Python loops over N

The inner "for each pair" loop processes N/2 pairs per stage. We implement this as a single vectorised operation on the entire array — no Python loop over individual elements.

In [ ]:
import jax
import jax.numpy as jnp
from sympy.discrete.transforms import ntt as sympy_ntt


def ntt(x, *, q, psi_powers, twiddles):
    """
    Forward negacyclic NTT.

    Args:
        x          : Input shape (batch, N), dtype uint32, values in [0, q)
        q          : Prime modulus (scalar uint32)
        psi_powers : Shape (N,) uint32 — psi_powers[n] = psi^n mod q
        twiddles   : Shape (N,) uint32 — Stockham twiddle table

    Returns:
        jnp.ndarray: NTT output, same shape (batch, N), dtype uint32
    """
    batch, N = x.shape
    stages = N.bit_length() - 1  # log2(N)

    # ── Step 1: Negacyclic twist ──────────────────────────────────────────────
    # Multiply each coefficient x[b, n] by psi_powers[n].
    # psi_powers shape: (N,) → broadcast over batch dim → (1, N)
    psi_row = psi_powers[None, :]  # shape (1, N) — broadcasts over batch
    out = mod_mul(x, psi_row, q)   # shape (batch, N) — elementwise, all at once

    # ── Step 2: Stockham cyclic NTT ───────────────────────────────────────────
    # log2(N) stages. Each stage is a single vectorised operation.
    for s in range(stages):
        span = 1 << s  # 2^s: grows from 1 to N/2

        # --- Extract twiddle factors for this stage ---
        # twiddles[span : 2*span] holds the `span` twiddle factors for this stage.
        # Reshape to (1, span) for broadcasting over (batch, N) arrays.
        W = twiddles[span : 2 * span]  # shape (span,)

        # --- Reshape out to separate pairs ---
        # View out as (batch, N/(2*span), 2, span):
        #   axis 0: batch
        #   axis 1: which block (there are N/(2*span) blocks)
        #   axis 2: upper (0) vs lower (1) half of each block
        #   axis 3: position within the half (0..span-1)
        num_blocks = N // (2 * span)
        out_4d = out.reshape(batch, num_blocks, 2, span)

        u = out_4d[:, :, 0, :]  # shape (batch, num_blocks, span) — upper halves
        v = out_4d[:, :, 1, :]  # shape (batch, num_blocks, span) — lower halves

        # W shape: (span,) → (1, 1, span) → broadcasts over (batch, num_blocks, span)
        W_bcast = W[None, None, :]  # shape (1, 1, span)

        # --- Apply butterfly to all pairs simultaneously ---
        Wv  = mod_mul(v, W_bcast, q)  # W * v  (batch, num_blocks, span)
        top = mod_add(u, Wv, q)        # u + Wv
        bot = mod_sub(u, Wv, q)        # u - Wv

        # --- Stockham reorder: interleave top/bot into output buffer ---
        # Stockham writes: top halves first (all blocks), then bot halves.
        # Reshape back: (batch, num_blocks, 2, span) → (batch, N)
        new_4d = jnp.stack([top, bot], axis=2)  # (batch, num_blocks, 2, span)
        out = new_4d.reshape(batch, N)

    return out


# ── Reference oracle ─────────────────────────────────────────────────────────
def negacyclic_ntt_oracle(x, *, q, psi):
    """Python reference using SymPy (slow but correct)."""
    x = [int(v) % q for v in x]
    N = len(x)
    psi_pow = [pow(psi, n, q) for n in range(N)]
    x_twisted = [(x[n] * psi_pow[n]) % q for n in range(N)]
    y = sympy_ntt(x_twisted, q)
    return [int(v) % q for v in y]


# ── Test on N=8 ───────────────────────────────────────────────────────────────
N_test = 8
q_test = 97
psi_test = find_primitive_root_of_unity(2 * N_test, q_test)

pp_np, tw_np = my_precompute_tables(N_test, q_test, psi_test)
pp = jnp.asarray(pp_np, dtype=jnp.uint32)
tw = jnp.asarray(tw_np, dtype=jnp.uint32)
Q  = jnp.uint32(q_test)

# Single row, then batch
x_single = jnp.array([[1, 2, 3, 4, 5, 6, 7, 8]], dtype=jnp.uint32)

y_mine = ntt(x_single, q=Q, psi_powers=pp, twiddles=tw)
y_ref  = negacyclic_ntt_oracle(x_single[0].tolist(), q=q_test, psi=psi_test)

print(f"Input    : {x_single[0].tolist()}")
print(f"My NTT   : {y_mine[0].tolist()}")
print(f"Reference: {y_ref}")
print(f"Correct  : {y_mine[0].tolist() == y_ref}")

# Batch test
x_batch = jnp.array([[1,2,3,4,5,6,7,8],[8,7,6,5,4,3,2,1],[0,1,0,1,0,1,0,1]], dtype=jnp.uint32)
y_batch = ntt(x_batch, q=Q, psi_powers=pp, twiddles=tw)
y_refs  = [negacyclic_ntt_oracle(x_batch[b].tolist(), q=q_test, psi=psi_test) for b in range(3)]

print(f"\nBatch test (3 rows):")
for b in range(3):
    match = y_batch[b].tolist() == y_refs[b]
    print(f"  Row {b}: {'OK' if match else 'FAIL'}")

### Linearity check

NTT is a linear transform: `NTT(a + b) = NTT(a) + NTT(b)` (mod q). This is a fast sanity check you can always run.

In [ ]:
import jax
import jax.numpy as jnp

key = jax.random.PRNGKey(42)
a = jax.random.randint(key, (1, N_test), 0, q_test, dtype=jnp.uint32)
b = jax.random.randint(key, (1, N_test), 0, q_test, dtype=jnp.uint32)

# NTT(a + b) mod q
ab_sum = (a.astype(jnp.uint64) + b.astype(jnp.uint64)).astype(jnp.uint32) % Q
lhs = ntt(ab_sum, q=Q, psi_powers=pp, twiddles=tw)

# NTT(a) + NTT(b) mod q
ya  = ntt(a, q=Q, psi_powers=pp, twiddles=tw)
yb  = ntt(b, q=Q, psi_powers=pp, twiddles=tw)
rhs = (ya.astype(jnp.uint64) + yb.astype(jnp.uint64)).astype(jnp.uint32) % Q

match = (lhs == rhs).all()
print(f"Linearity: NTT(a+b) == NTT(a)+NTT(b) ?  {match}")

---
## Section 9 — GPU and TPU Optimisation

### 9.1 — What ops are hot?

Inside each Stockham stage, every element of the array participates in exactly one butterfly. With N = 2^12 = 4096 and 4 batches, that's 4096 × 4 = 16384 butterflies per stage, each with 1 `mod_mul` + 1 `mod_add` + 1 `mod_sub`.

| Operation | Parallelisable? | GPU/TPU benefit | Notes |
|---|---|---|---|
| Twist: `mod_mul(x, psi_powers)` | **YES** | Very large | N×batch elementwise, fully parallel |
| Butterfly `mod_mul + mod_add + mod_sub` per stage | **YES** | Very large | N/2×batch per stage, all independent |
| Reshape / transpose (`out.reshape`) | **YES** | Free | No compute, just metadata |
| `jnp.stack([top, bot])` | **YES** | Large | Contiguous write |
| Outer Python `for s in range(stages)` | **NO** | None | `stages = log2(N)` ≤ 12 iterations — small |

The **bottleneck is always the butterflies inside each stage** — and they're all independent, so GPUs handle them perfectly.

### 9.2 — `jax.jit`: compile the entire function once

In [ ]:
import jax
import jax.numpy as jnp
import time

# Use a realistic size: N = 2^12, batch = 4
N_big   = 1 << 12  # 4096
BATCH   = 4

# Find a real NTT prime for this N
# q = 1 mod 2N = 1 mod 8192.  First such prime below 2^31:
# We search: 2^31 - 1 = 2147483647. (2147483647 - 1) // 8192 = 262143. 262143*8192+1 = 2147475457. Check isprime...
from sympy import isprime
step = 2 * N_big
candidate = (((1 << 31) - 1 - 1) // step) * step + 1
while not isprime(candidate):
    candidate -= step
q_big  = candidate
psi_big = find_primitive_root_of_unity(2 * N_big, q_big)

pp_big_np, tw_big_np = my_precompute_tables(N_big, q_big, psi_big)
pp_big = jnp.asarray(pp_big_np, dtype=jnp.uint32)
tw_big = jnp.asarray(tw_big_np, dtype=jnp.uint32)
Q_big  = jnp.uint32(q_big)

key = jax.random.PRNGKey(0)
x_big = jax.random.randint(key, (BATCH, N_big), 0, q_big, dtype=jnp.uint32)

print(f"N={N_big}, batch={BATCH}, q={q_big}")
print(f"Stages = log2({N_big}) = {N_big.bit_length()-1}")

# ── Eager (no JIT) ───────────────────────────────────────────────────────────
_ = ntt(x_big, q=Q_big, psi_powers=pp_big, twiddles=tw_big).block_until_ready()  # warm-up

RUNS = 10
t0 = time.perf_counter()
for _ in range(RUNS):
    ntt(x_big, q=Q_big, psi_powers=pp_big, twiddles=tw_big).block_until_ready()
t_eager = (time.perf_counter() - t0) / RUNS * 1000

# ── JIT compiled ─────────────────────────────────────────────────────────────
ntt_jit = jax.jit(lambda z: ntt(z, q=Q_big, psi_powers=pp_big, twiddles=tw_big))

# First call triggers compilation
t_compile_start = time.perf_counter()
_ = ntt_jit(x_big).block_until_ready()
t_compile = (time.perf_counter() - t_compile_start) * 1000

t0 = time.perf_counter()
for _ in range(RUNS):
    ntt_jit(x_big).block_until_ready()
t_jit = (time.perf_counter() - t0) / RUNS * 1000

throughput = BATCH * N_big / (t_jit / 1000) / 1e6

print(f"\nEager (no JIT):    {t_eager:.2f} ms/call")
print(f"JIT compile time:  {t_compile:.1f} ms (one-time cost)")
print(f"JIT steady-state:  {t_jit:.3f} ms/call")
print(f"Speedup from JIT:  {t_eager/t_jit:.1f}x")
print(f"Throughput:        {throughput:.2f} Mcoeff/s")
print(f"Backend:           {jax.default_backend()}")

### 9.3 — Memory layout: why the reshape matters on GPU

The Stockham reshape `out.reshape(batch, num_blocks, 2, span)` is free (no data copy). But it makes the memory access pattern for the butterfly perfectly coalesced:

- All threads in a GPU warp access consecutive memory addresses.
- No bank conflicts in shared memory.
- XLA/CUDA can fuse the butterfly into a single kernel with no intermediate writes to global memory.

Compare this to a naive in-place bit-reversal permutation (Cooley-Tukey style), which causes stride-N memory accesses — catastrophic for GPU cache lines.

In [ ]:
import jax.numpy as jnp

# Demonstrate the reshape trick on a small array
# At stage s=1 (span=2), N=8, batch=1:
# out shape: (1, 8)
# num_blocks = 8 / (2*2) = 2
# reshape to (1, 2, 2, 2) = (batch, num_blocks, 2, span)

batch, N_demo, span, s = 1, 8, 2, 1
num_blocks = N_demo // (2 * span)

out = jnp.arange(8, dtype=jnp.uint32).reshape(1, N_demo)  # values 0..7
print(f"Flat array (batch=1, N={N_demo}): {out[0].tolist()}")
print(f"                  indices:   {list(range(N_demo))}")

out_4d = out.reshape(batch, num_blocks, 2, span)
print(f"\nReshaped to (batch={batch}, num_blocks={num_blocks}, 2, span={span}):")
for blk in range(num_blocks):
    u = out_4d[0, blk, 0, :].tolist()  # upper half of this block
    v = out_4d[0, blk, 1, :].tolist()  # lower half of this block
    print(f"  Block {blk}: u (upper) = {u},  v (lower) = {v}")

print("\nGPU sees each block's u and v as consecutive in memory →")
print("all threads process their pair without cache conflicts.")

### 9.4 — Montgomery multiplication: the advanced optimisation

The `%` operator (modular reduction) hides a division, which is slow on both CPU and GPU. **Montgomery multiplication** replaces the modular reduction with cheap bit shifts and additions, at the cost of a one-time conversion step.

#### How it works

Choose `R = 2^32` (or any power of 2 that is coprime to q).  
**Montgomery form** of `a` is `ā = a·R mod q`.

Montgomery multiplication computes:
```
MontMul(ā, b̄) = ā·b̄·R⁻¹ mod q  =  (a·b)·R mod q
```
The key: reducing `mod q` is replaced by:
1. A multiply by `q_inv` (mod R) — just a bit-AND with R-1.
2. A comparison and conditional subtract — no division.

#### Trade-off
- Cost: convert psi_powers and twiddles to Montgomery form in `prepare_tables` (done once, not timed).
- Benefit: every `mod_mul` in the hot NTT loop is ~2–4x faster on GPU.
- Worth it for N ≥ 2^10 where you're doing many multiplies.

In [ ]:
import jax.numpy as jnp
import numpy as np

# ── Montgomery multiplication (conceptual walkthrough) ───────────────────────
# We use R = 2^32 so that all arithmetic fits in uint64.

def precompute_montgomery_params(q):
    """
    Compute Montgomery reduction parameters for modulus q.

    Returns:
        q_inv: -q^{-1} mod R  (used in reduction step)
        R_sq : R^2 mod q      (used to convert a → Montgomery form a*R mod q)
    """
    R = 1 << 32  # R = 2^32
    # q_inv = -q^{-1} mod R  (extended Euclidean algorithm)
    q_inv = pow(q, -1, R)   # Python 3.8+ supports modular inverse directly
    q_inv = (-q_inv) % R    # negate to get -q^{-1} mod R
    R_sq  = (R * R) % q     # R^2 mod q (fits in uint64 since q < 2^31)
    return q_inv, R_sq


def mont_mul(a, b, q, q_inv):
    """
    Montgomery multiplication: returns a*b*R^{-1} mod q.
    Inputs a, b should already be in Montgomery form (i.e., a = x*R mod q).

    For two inputs in Montgomery form:
      MontMul(a*R, b*R) = a*b*R mod q  (still in Montgomery form)

    Uses uint64 arithmetic throughout. R = 2^32.
    """
    R = jnp.uint64(1 << 32)
    q64     = jnp.uint64(q)
    q_inv64 = jnp.uint64(q_inv)

    a64 = a.astype(jnp.uint64)
    b64 = b.astype(jnp.uint64)

    t    = a64 * b64                          # t = a*b  (up to 2^62)
    m    = (t * q_inv64) % R                  # m = t * (-q^{-1}) mod R  (low 32 bits)
    u    = (t + m * q64) >> jnp.uint64(32)    # u = (t + m*q) / R  (shift out low 32 bits)
    # Conditional subtract to bring result into [0, q)
    result = jnp.where(u >= q64, u - q64, u)
    return result.astype(jnp.uint32)


def to_montgomery(a, q, R_sq, q_inv):
    """Convert a to Montgomery form: returns a*R mod q."""
    return mont_mul(a, jnp.asarray(R_sq, dtype=jnp.uint32), q, q_inv)


def from_montgomery(a_mont, q, q_inv):
    """Convert from Montgomery form back to standard form: returns a*R^{-1} mod q."""
    one = jnp.ones_like(a_mont)
    return mont_mul(a_mont, one, q, q_inv)


# ── Verify correctness ───────────────────────────────────────────────────────
q_val = 97
q_inv_val, R_sq_val = precompute_montgomery_params(q_val)
print(f"q={q_val}, q_inv (-q^{{-1}} mod 2^32) = {q_inv_val}, R^2 mod q = {R_sq_val}")

a_vals = jnp.array([5, 10, 23, 96], dtype=jnp.uint32)
b_vals = jnp.array([3,  7, 41, 50], dtype=jnp.uint32)

# Convert to Montgomery form
a_mont = to_montgomery(a_vals, q_val, R_sq_val, q_inv_val)
b_mont = to_montgomery(b_vals, q_val, R_sq_val, q_inv_val)

# Multiply in Montgomery domain
result_mont = mont_mul(a_mont, b_mont, q_val, q_inv_val)

# Convert result back to standard form
result = from_montgomery(result_mont, q_val, q_inv_val)

# Verify against standard mod_mul
expected = mod_mul(a_vals, b_vals, jnp.uint32(q_val))
print(f"\na = {a_vals.tolist()}")
print(f"b = {b_vals.tolist()}")
print(f"Montgomery a*b mod q = {result.tolist()}")
print(f"Standard   a*b mod q = {expected.tolist()}")
print(f"Match: {result.tolist() == expected.tolist()}")

### 9.5 — vmap: free batching

Instead of writing batch-handling logic manually (the `batch` dimension in our loops), `jax.vmap` can automatically vectorise a single-row NTT over a batch. This:
- Simplifies code (write NTT for a single row, vmap handles batching).
- On GPU, vmap fuses the batch into the kernel — no extra Python overhead.
- Equivalent performance to our explicit `(batch, N)` formulation.

In [ ]:
import jax
import jax.numpy as jnp
import time

# Single-row NTT (shape (1, N) input)
def ntt_single_row(x_row, *, q, psi_powers, twiddles):
    """NTT for a single polynomial: x_row shape (N,), returns shape (N,)."""
    return ntt(x_row[None, :], q=q, psi_powers=psi_powers, twiddles=twiddles)[0]


# vmap over the batch dimension
ntt_vmapped = jax.vmap(
    lambda row: ntt_single_row(row, q=Q_big, psi_powers=pp_big, twiddles=tw_big)
)

# JIT the vmapped version
ntt_vmapped_jit = jax.jit(ntt_vmapped)

# Warm-up
_ = ntt_vmapped_jit(x_big).block_until_ready()

RUNS = 10
t0 = time.perf_counter()
for _ in range(RUNS):
    ntt_vmapped_jit(x_big).block_until_ready()
t_vmap = (time.perf_counter() - t0) / RUNS * 1000

# Also JIT the direct batched version for comparison
ntt_direct_jit = jax.jit(lambda z: ntt(z, q=Q_big, psi_powers=pp_big, twiddles=tw_big))
_ = ntt_direct_jit(x_big).block_until_ready()
t0 = time.perf_counter()
for _ in range(RUNS):
    ntt_direct_jit(x_big).block_until_ready()
t_direct = (time.perf_counter() - t0) / RUNS * 1000

print(f"Direct batched NTT (JIT): {t_direct:.3f} ms/call")
print(f"vmap batched NTT  (JIT): {t_vmap:.3f} ms/call")
print("Both should be similar — vmap is just syntactic sugar here.")

# Correctness check
y_direct_out = ntt_direct_jit(x_big)
y_vmap_out   = ntt_vmapped_jit(x_big)
print(f"\nOutputs match: {(y_direct_out == y_vmap_out).all()}")

### 9.6 — GPU vs CPU: concrete comparison

Run this cell to see side-by-side timing across all available devices.

In [ ]:
import jax
import jax.numpy as jnp
import time

print("Devices:", jax.devices())

ntt_fn = jax.jit(lambda z: ntt(z, q=Q_big, psi_powers=pp_big, twiddles=tw_big))

for device in jax.devices():
    x_dev  = jax.device_put(x_big, device)
    pp_dev = jax.device_put(pp_big, device)
    tw_dev = jax.device_put(tw_big, device)

    fn = jax.jit(lambda z: ntt(z, q=Q_big, psi_powers=pp_dev, twiddles=tw_dev))

    # Warm-up (triggers JIT)
    _ = fn(x_dev).block_until_ready()

    RUNS = 20
    t0 = time.perf_counter()
    for _ in range(RUNS):
        fn(x_dev).block_until_ready()
    elapsed_ms = (time.perf_counter() - t0) / RUNS * 1000

    throughput = BATCH * N_big / (elapsed_ms / 1000) / 1e6
    print(f"{str(device.device_kind):30s}  {elapsed_ms:.3f} ms  {throughput:.2f} Mcoeff/s")

### 9.7 — Optimisation summary

```
┌──────────────────────────────────────────────────────────────────────────┐
│                      NTT OPTIMISATION GUIDE                              │
├─────────────────────────┬───────────────────────────┬────────────────────┤
│  Operation              │  How to optimise           │  Where it helps    │
├─────────────────────────┼───────────────────────────┼────────────────────┤
│ mod_mul in butterfly    │ Montgomery multiplication  │ GPU: 2-4x speedup  │
│                         │ in prepare_tables          │ CPU: 1.5-2x        │
├─────────────────────────┼───────────────────────────┼────────────────────┤
│ All butterfly ops       │ jax.jit entire ntt()       │ GPU: huge          │
│                         │ Fuses all stage kernels    │ CPU: moderate      │
├─────────────────────────┼───────────────────────────┼────────────────────┤
│ Memory access pattern   │ Stockham out-of-place      │ GPU: huge          │
│                         │ (already in template)      │ (coalesced access) │
├─────────────────────────┼───────────────────────────┼────────────────────┤
│ Batch dimension         │ Keep (batch, N) layout     │ GPU: free          │
│                         │ or use jax.vmap            │ Both backends      │
├─────────────────────────┼───────────────────────────┼────────────────────┤
│ Table precomputation    │ put in prepare_tables()    │ Not timed: free!   │
│ (psi_powers, twiddles)  │ convert to Montgomery form │                    │
├─────────────────────────┼───────────────────────────┼────────────────────┤
│ log2(N) stage loop      │ Cannot parallelise (data   │ N/A                │
│                         │ dependency between stages) │                    │
├─────────────────────────┼───────────────────────────┼────────────────────┤
│ Twist (N mod_muls)      │ Already in hot path; JIT   │ GPU: large         │
│                         │ fuses with first stage     │                    │
└─────────────────────────┴───────────────────────────┴────────────────────┘
```

---
## Section 10 — `prepare_tables` and Proper Benchmarking

### prepare_tables: the free optimisation slot

The benchmark harness calls `prepare_tables` **before** starting the timer. This is your chance to:
- Convert psi_powers and twiddles to Montgomery form (big win).
- Re-layout the twiddle table to match your exact indexing scheme.
- Pre-broadcast or pad tables for your chosen reshape strategy.

All that work is excluded from the latency measurement.

In [ ]:
import jax.numpy as jnp
import numpy as np


def prepare_tables_montgomery(*, q, psi_powers, twiddles):
    """
    Convert psi_powers and twiddles to Montgomery form.
    Called once before timing — cost is not counted.

    Returns (psi_powers_mont, twiddles_mont) ready for use in ntt_montgomery().
    """
    q_int = int(q)
    q_inv, R_sq = precompute_montgomery_params(q_int)

    pp_mont = to_montgomery(psi_powers, q_int, R_sq, q_inv)
    tw_mont = to_montgomery(twiddles,   q_int, R_sq, q_inv)

    # Return also q_inv so the ntt function can use it
    # Wrap in a dict for clarity (or return as a tuple matching the API)
    return pp_mont, tw_mont


def ntt_with_montgomery(x, *, q, psi_powers, twiddles, q_inv):
    """
    NTT using Montgomery multiplication throughout.
    psi_powers and twiddles must already be in Montgomery form.
    """
    batch, N = x.shape
    stages   = N.bit_length() - 1
    q_int    = int(q)
    q_inv_int= int(q_inv)

    # Convert x to Montgomery form once
    _, R_sq = precompute_montgomery_params(q_int)
    out = to_montgomery(x, q_int, R_sq, q_inv_int)

    # Twist in Montgomery domain
    psi_row = psi_powers[None, :]  # (1, N)
    out = mont_mul(out, psi_row, q_int, q_inv_int)

    # Stockham stages using Montgomery multiplication
    for s in range(stages):
        span       = 1 << s
        num_blocks = N // (2 * span)
        W          = twiddles[span : 2 * span]

        out_4d = out.reshape(batch, num_blocks, 2, span)
        u = out_4d[:, :, 0, :]
        v = out_4d[:, :, 1, :]

        W_bcast = W[None, None, :]

        # Montgomery multiply: no % needed!
        Wv  = mont_mul(v, W_bcast, q_int, q_inv_int)
        top = mod_add(u, Wv, jnp.uint32(q_int))
        bot = mod_sub(u, Wv, jnp.uint32(q_int))

        new_4d = jnp.stack([top, bot], axis=2)
        out = new_4d.reshape(batch, N)

    # Convert back from Montgomery form
    out = from_montgomery(out, q_int, q_inv_int)
    return out


# ── Verify correctness of Montgomery NTT ─────────────────────────────────────
q_test2 = 97
N_test2 = 8
psi_test2 = find_primitive_root_of_unity(2 * N_test2, q_test2)
pp2_np, tw2_np = my_precompute_tables(N_test2, q_test2, psi_test2)
pp2 = jnp.asarray(pp2_np, dtype=jnp.uint32)
tw2 = jnp.asarray(tw2_np, dtype=jnp.uint32)
Q2  = jnp.uint32(q_test2)

# Prepare Montgomery tables
q_inv2, R_sq2  = precompute_montgomery_params(q_test2)
pp2_mont = to_montgomery(pp2, q_test2, R_sq2, q_inv2)
tw2_mont = to_montgomery(tw2, q_test2, R_sq2, q_inv2)

x_test2 = jnp.array([[1, 2, 3, 4, 5, 6, 7, 8]], dtype=jnp.uint32)
y_std   = ntt(x_test2, q=Q2, psi_powers=pp2, twiddles=tw2)
y_mont  = ntt_with_montgomery(x_test2, q=Q2, psi_powers=pp2_mont, twiddles=tw2_mont, q_inv=q_inv2)

print("Standard NTT result:    ", y_std[0].tolist())
print("Montgomery NTT result:  ", y_mont[0].tolist())
print("Match:", y_std[0].tolist() == y_mont[0].tolist())

### Benchmarking correctly with JAX

Three common mistakes when timing JAX code:

1. **Not calling `.block_until_ready()`** — JAX dispatches work asynchronously; Python returns immediately while the GPU is still computing. Without blocking, you measure dispatch latency, not compute latency.
2. **Timing the first call** — the first JIT call triggers XLA compilation, which can take seconds. Always do warmup runs and time only after compilation.
3. **Not accounting for data transfer** — if data is on CPU and computation on GPU, the first run may include PCIe transfer time.

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import time


def benchmark_ntt(logn, batch, warmup=5, runs=20):
    """
    Benchmark the NTT following the same pattern as tests/benchmark.py.

    logn  : log2(N)
    batch : batch size (B)
    warmup: number of warmup calls (first includes JIT compile)
    runs  : number of timed calls
    """
    N   = 1 << logn

    # Find NTT prime for this N
    from sympy import isprime
    step = 2 * N
    candidate = (((1 << 31) - 1 - 1) // step) * step + 1
    while not isprime(candidate):
        candidate -= step
    q_val = candidate

    psi_val = find_primitive_root_of_unity(2 * N, q_val)

    # Precompute tables (not timed)
    pp_np, tw_np = my_precompute_tables(N, q_val, psi_val)
    pp = jnp.asarray(pp_np, dtype=jnp.uint32)
    tw = jnp.asarray(tw_np, dtype=jnp.uint32)
    Q  = jnp.uint32(q_val)

    # Random input on the target device
    key = jax.random.PRNGKey(42)
    x = jax.random.randint(key, (batch, N), 0, q_val, dtype=jnp.uint32)

    # JIT-compile the function
    fn = jax.jit(lambda z: ntt(z, q=Q, psi_powers=pp, twiddles=tw))

    # First call: measure JIT compile time
    t0 = time.perf_counter()
    _ = fn(x).block_until_ready()
    compile_ms = (time.perf_counter() - t0) * 1000

    # Remaining warmup
    for _ in range(warmup - 1):
        fn(x).block_until_ready()

    # Timed runs
    times_ms = []
    for _ in range(runs):
        t0 = time.perf_counter()
        fn(x).block_until_ready()  # MUST block for accurate timing
        times_ms.append((time.perf_counter() - t0) * 1000)

    times_arr = np.array(times_ms)
    median_ms = float(np.median(times_arr))
    p90_ms    = float(np.quantile(times_arr, 0.90))
    throughput = batch * N / (median_ms / 1000) / 1e6

    print(f"log2(N)={logn:2d}  N={N:6d}  batch={batch}  "
          f"compile={compile_ms:.0f}ms  "
          f"median={median_ms:.3f}ms  "
          f"p90={p90_ms:.3f}ms  "
          f"{throughput:.2f} Mcoeff/s  "
          f"[{jax.default_backend()}]")
    return median_ms


print(f"{'logN':>6}  {'N':>7}  {'batch':>5}  {'compile':>10}  {'median':>9}  {'p90':>9}  {'Mcoeff/s':>10}  backend")
print("-" * 80)
for logn in [8, 10, 12]:
    benchmark_ntt(logn, batch=4, warmup=5, runs=20)

---
## Quick Reference

### API the harness calls
```python
# Called once before timing:
psi_powers, twiddles = prepare_tables(q=q, psi_powers=psi_powers, twiddles=twiddles)

# Called in the hot loop (this is what gets timed):
y = ntt(x, q=q, psi_powers=psi_powers, twiddles=twiddles)
#  x shape: (batch, N)  dtype: uint32
#  y shape: (batch, N)  dtype: uint32  values in [0, q)
```

### NTT at a glance
```
Step 1 — Twist:      x_twisted[n] = x[n] · psi_powers[n]  (mod q)
Step 2 — log2(N) Stockham stages:
   for s = 0 .. log2(N)-1:
       span = 2^s
       for each block of 2*span elements:
           for j = 0..span-1:
               W   = twiddles[span + j]
               u   = current[block_start + j]
               v   = current[block_start + span + j]
               top = (u + W*v) mod q
               bot = (u - W*v) mod q
```

### Key identities
```
ψ^{2N} ≡  1  (mod q)   ← psi is a 2N-th root of unity
ψ^N    ≡ -1  (mod q)   ← negacyclic property
ω = ψ²                 ← omega is the N-th root for the cyclic part
q ≡ 1 (mod 2N)         ← required for psi to exist mod q
```

### GPU performance checklist
1. **JIT** the entire `ntt()` function — fuses all stage kernels into one.
2. **Montgomery** form in `prepare_tables` — replace `%` with cheap bit ops in the hot path.
3. **Stockham** layout — already in the template; don't switch to in-place Cooley-Tukey.
4. **Batch** operations — keep `(batch, N)` arrays; never loop over batch in Python.
5. **block_until_ready()** in benchmarks — JAX is async; always block before measuring.
6. Stage loop (`log2(N)` iterations) is sequential — focus optimisation on the per-stage butterfly.

### Useful links
- NTT Tutorial: https://eprint.iacr.org/2024/585.pdf
- Montgomery Multiplication: https://cp-algorithms.com/algebra/montgomery_multiplication.html
- Fast GPU NTT #1: https://arxiv.org/pdf/2012.01968
- Fast GPU NTT #2: https://arxiv.org/pdf/2407.13055
- JAX JIT docs: https://jax.readthedocs.io/en/latest/jit-compilation.html